# Mini-DDSM — IEEE CSDE 2026 Final Local Reviewer-Revision Notebook

**Paper:** *Multi-Backbone Ensemble Learning for Robust Breast Cancer Classification on the Mini-DDSM Mammography Dataset*

This local notebook is aligned with the **submitted paper's primary model definition** while adding the analyses requested by the reviewers.

### Primary method retained from the submitted paper
- 3-fold patient-grouped outer cross-validation.
- EfficientNet-B4 and ConvNeXtV2-Tiny.
- Four-view TTA: original, horizontal flip, vertical flip, 180° rotation.
- Image predictions are averaged within each backbone, then **mean-aggregated to one score per patient**.
- The primary two-backbone ensemble is **unweighted 50/50 probability averaging**.
- EfficientNet-B4: LR `2e-4`, maximum 25 epochs, early-stopping patience 6.
- ConvNeXtV2-Tiny: LR `5e-5`, maximum 17 epochs, early-stopping patience 4.

### Reviewer-requested additions
- exact patient-ID collision audit and corrected split files;
- 3-fold experiment repeated over multiple random seeds;
- patient/image counts and class balance per fold;
- strict inner-validation separation for checkpoint selection and threshold tuning;
- sensitivity, specificity, PPV, NPV, accuracy, ROC-AUC, PR-AUC, confidence intervals;
- default 0.5 and train-side Youden thresholds;
- identical-TTA comparison of both single backbones, inner-selected best single backbone, submitted equal-average ensemble, and an **exploratory** validation-weighted ensemble;
- calibration metrics/plots and decision-curve analysis;
- false-negative malignant-case analysis;
- optional external validation template for CBIS-DDSM or INbreast.

**Important methodological correction for the camera-ready paper:** threshold selection is performed only on the inner validation patients of each outer training fold. Outer-test labels are never used for checkpoint selection, ensemble weighting, calibration, or Youden-threshold selection.

## 0. Local environment and configuration

This version is intended for Ubuntu/Linux and an NVIDIA GPU. Edit `DATA_ROOT` if auto-detection does not find the Mini-DDSM folder. The official dataset used by the submitted paper should contain **9,684 images**: 2,728 Normal, 3,360 Benign, and 3,596 Cancer.

In [1]:
import os, gc, glob, json, math, random, hashlib, warnings, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import timm
except ImportError as e:
    raise ImportError(
        "Missing package 'timm'. In your Ubuntu terminal run: pip install timm"
    ) from e

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

try:
    from scipy.stats import t as student_t
except Exception:
    student_t = None

from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- Local dataset path ----------------
# You may set an environment variable instead:
#   export MINI_DDSM_ROOT=/full/path/to/Mini_DDSM_Upload
_env_root = os.environ.get("MINI_DDSM_ROOT", "").strip()
_candidates = [
    Path(_env_root).expanduser() if _env_root else None,
    Path("/Research/MINI_DDSM/archive (1)/Mini_DDSM_Upload"),
    Path.cwd() / "archive (1)" / "Mini_DDSM_Upload",
    Path.cwd() / "Mini_DDSM_Upload",
    Path("/Research/MINI_DDSM/Mini_DDSM_Upload"),
    Path.home() / "Datasets" / "Mini_DDSM_Upload",
    Path.home() / "data" / "Mini_DDSM_Upload",
    Path.home() / "Downloads" / "Mini_DDSM_Upload",
    Path.home() / "Mini_DDSM_Upload",
]
_candidates = [p for p in _candidates if p is not None]
DATA_ROOT = next((p for p in _candidates if p.exists()), _candidates[0])

# ---------------- Core paper-aligned settings ----------------
IMG_SIZE = 512
N_FOLDS = 3
SEEDS = [42, 123, 2026]
INNER_VAL_FRACTION = 0.12

BACKBONES = ["tf_efficientnet_b4", "convnextv2_tiny"]
BACKBONE_LR = {
    "tf_efficientnet_b4": 2e-4,
    "convnextv2_tiny": 5e-5,
}
BACKBONE_MAX_EPOCHS = {
    "tf_efficientnet_b4": 25,
    "convnextv2_tiny": 17,
}
BACKBONE_PATIENCE = {
    "tf_efficientnet_b4": 6,
    "convnextv2_tiny": 4,
}
EARLY_STOP_MIN_DELTA = 0.0

# Safe default for 12 GB VRAM at 512x512.
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2  # effective batch ~= 16
NUM_WORKERS = min(4, os.cpu_count() or 1)

PATIENT_AGG = "mean"
WEIGHT_GRID = np.round(np.linspace(0.10, 0.90, 17), 2)
N_BOOT = 2000
BOOTSTRAP_SEED = 2709

RUN_DUPLICATE_HASH_AUDIT = True
FAIL_ON_CROSS_PATIENT_EXACT_DUPLICATE = False
STRICT_OFFICIAL_DATASET_COUNTS = True

EXPECTED_COUNTS = {
    "Normal": 2728,
    "Benign": 3360,
    "Cancer": 3596,
}
EXPECTED_TOTAL = 9684

OUTPUT_DIR = Path.cwd() / "mini_ddsm_csde2026_final_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Optional external validation.
ENABLE_EXTERNAL_VALIDATION = False
EXTERNAL_NAME = "CBIS-DDSM"  # or "INbreast"
EXTERNAL_CSV = Path.home() / "Datasets" / "external_manifest.csv"

# ---------------- Device / reproducibility ----------------
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEVICE = get_device()

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / (1024**3):.1f} GB")
else:
    print("WARNING: CUDA GPU not detected; training will be very slow on CPU.")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)
print("Outputs:", OUTPUT_DIR)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset, batch_size, shuffle, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=g,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
    return DataLoader(**kwargs)

seed_everything(SEEDS[0])

/home/ice/miniforge3/envs/ml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 5070
VRAM: 11.5 GB
PyTorch: 2.13.0+cu132
Device: cuda
DATA_ROOT: /Research/MINI_DDSM/archive (1)/Mini_DDSM_Upload
Outputs: /Research/MINI_DDSM/mini_ddsm_csde2026_final_outputs


## 1. Build the Mini-DDSM manifest and reproduce the patient-ID correction

The raw filename-derived ID is preserved as `raw_patient_id`. The corrected grouping key is `class_folder::raw_patient_id`, matching the submitted paper's collision-correction rule. Any images sharing the corrected patient ID—bilateral views, repeated views, or additional files—remain in the same outer fold.

In [2]:
LABEL_MAP = {"Normal": 0, "Benign": 0, "Cancer": 1}


def parse_filename(path, cls_folder):
    name = os.path.basename(path)
    parts = name.split(".")
    id_part = parts[0]
    raw_patient_id = id_part.split("_")[0]
    patient_id = f"{cls_folder}::{raw_patient_id}"

    side_view = parts[1] if len(parts) > 1 else ""
    upper = side_view.upper()
    side = "L" if upper.startswith("LEFT") else ("R" if upper.startswith("RIGHT") else "UNK")
    view = "CC" if "CC" in upper else ("MLO" if "MLO" in upper else "UNK")
    return str(raw_patient_id), str(patient_id), side, view


def discover_images(root):
    root = Path(root).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(
            f"Mini-DDSM folder not found: {root}\n"
            "Edit DATA_ROOT in Cell 0 or set MINI_DDSM_ROOT in your shell."
        )

    rows = []
    for cls_folder, label in LABEL_MAP.items():
        cls_path = root / cls_folder
        if not cls_path.is_dir():
            raise FileNotFoundError(f"Missing class folder: {cls_path}")

        paths = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff", "*.dcm"):
            paths.extend(cls_path.glob(ext))

        for p in sorted(paths):
            raw_pid, pid, side, view = parse_filename(str(p), cls_folder)
            rows.append({
                "path": str(p),
                "filename": p.name,
                "raw_patient_id": raw_pid,
                "patient_id": pid,
                "class_folder": cls_folder,
                "label": int(label),
                "side": side,
                "view": view,
            })

    return pd.DataFrame(rows)


df = discover_images(DATA_ROOT)
if df.empty:
    raise RuntimeError("No images found in Mini-DDSM.")

# Official dataset count check against the submitted paper.
class_counts = df["class_folder"].value_counts().to_dict()
print("Class image counts:", class_counts)
print("Total images:", len(df))

if STRICT_OFFICIAL_DATASET_COUNTS:
    if len(df) != EXPECTED_TOTAL:
        raise RuntimeError(
            f"Expected {EXPECTED_TOTAL} images but found {len(df)}. "
            "Use the same Mini-DDSM version as the submitted paper."
        )
    for cls, expected in EXPECTED_COUNTS.items():
        got = int(class_counts.get(cls, 0))
        if got != expected:
            raise RuntimeError(f"Expected {expected} {cls} images but found {got}.")

# Every corrected patient must map to exactly one binary label.
label_nunique = df.groupby("patient_id")["label"].nunique()
if (label_nunique > 1).any():
    bad = label_nunique[label_nunique > 1].index.tolist()
    raise RuntimeError(f"Corrected patient IDs with inconsistent labels: {bad[:10]}")

# Detect raw-ID collisions across class namespaces.
collision_table = (
    df.groupby("raw_patient_id")
      .agg(
          n_class_folders=("class_folder", "nunique"),
          class_folders=("class_folder", lambda x: "|".join(sorted(set(x)))),
          n_images=("path", "size"),
      )
      .reset_index()
)
collision_table = collision_table[collision_table["n_class_folders"] > 1].copy()

# Patient inventory.
patient_inventory = (
    df.groupby("patient_id")
      .agg(
          label=("label", "max"),
          class_folder=("class_folder", "first"),
          n_images=("path", "size"),
          n_sides=("side", lambda x: len({v for v in x if v != "UNK"})),
          n_views=("view", lambda x: len({v for v in x if v != "UNK"})),
      )
      .reset_index()
)

print("Corrected unique patients:", df["patient_id"].nunique())
print("Raw IDs colliding across class folders:", len(collision_table))
print("Patients with >4 files:", int((patient_inventory["n_images"] > 4).sum()))

df.to_csv(OUTPUT_DIR / "corrected_patient_manifest.csv", index=False)
collision_table.to_csv(OUTPUT_DIR / "raw_patient_id_collisions.csv", index=False)
patient_inventory.to_csv(OUTPUT_DIR / "patient_inventory.csv", index=False)

display(collision_table.head(20))
display(
    patient_inventory.groupby("label")
    .agg(patients=("patient_id", "nunique"), images=("n_images", "sum"))
    .rename(index={0: "Not-Cancer", 1: "Cancer"})
)

Class image counts: {'Cancer': 3596, 'Benign': 3360, 'Normal': 2728}
Total images: 9684
Corrected unique patients: 9684
Raw IDs colliding across class folders: 3364
Patients with >4 files: 0


,raw_patient_id,n_class_folders,class_folders,n_images
0,1,3,Benign|Cancer|Normal,3
1,10,3,Benign|Cancer|Normal,3
2,100,3,Benign|Cancer|Normal,3
3,1000,3,Benign|Cancer|Normal,3
4,1001,3,Benign|Cancer|Normal,3
5,1002,3,Benign|Cancer|Normal,3
6,1003,3,Benign|Cancer|Normal,3
7,1004,3,Benign|Cancer|Normal,3
8,1005,3,Benign|Cancer|Normal,3
9,1006,3,Benign|Cancer|Normal,3


,patients,images
label,,
Not-Cancer,6088,6088
Cancer,3596,3596


## 2. Exact duplicate audit

SHA-256 detects exact byte-identical files. The audit does not pretend to detect visually similar near-duplicates. Exact duplicates are exported. If an identical file appears under different corrected patients, the notebook stops because that can create cross-patient leakage. Exact duplicates within the same patient are reported rather than silently removed, preserving the submitted dataset inventory unless you explicitly decide otherwise in the camera-ready revision.

In [3]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


if RUN_DUPLICATE_HASH_AUDIT:
    hash_cache = OUTPUT_DIR / "file_hashes.csv"
    hashes = None

    if hash_cache.exists():
        cached = pd.read_csv(hash_cache)
        if set(cached["path"].astype(str)) == set(df["path"].astype(str)):
            hashes = cached

    if hashes is None:
        hashes = df[["path", "patient_id", "label", "class_folder"]].copy()
        hashes["sha256"] = [sha256_file(p) for p in hashes["path"]]
        hashes.to_csv(hash_cache, index=False)

    exact_dups = hashes.groupby("sha256").filter(lambda g: len(g) > 1).copy()
    cross_patient_dups = exact_dups.groupby("sha256").filter(
        lambda g: g["patient_id"].nunique() > 1
    ).copy()

    exact_dups.to_csv(OUTPUT_DIR / "exact_duplicate_files.csv", index=False)
    cross_patient_dups.to_csv(OUTPUT_DIR / "cross_patient_exact_duplicates.csv", index=False)

    print("Rows participating in exact duplicate groups:", len(exact_dups))
    print("Rows in cross-patient exact duplicate groups:", len(cross_patient_dups))

    if FAIL_ON_CROSS_PATIENT_EXACT_DUPLICATE and len(cross_patient_dups) > 0:
        raise RuntimeError(
            "Exact image bytes occur under different corrected patient IDs. "
            "Inspect cross_patient_exact_duplicates.csv before training."
        )
else:
    print("Duplicate hash audit disabled.")

Rows participating in exact duplicate groups: 28
Rows in cross-patient exact duplicate groups: 28


## 3. Preprocessing, augmentation, model, TTA, and patient aggregation

The submitted paper uses Otsu-based breast-region cropping, resize to 512×512, grayscale-to-3-channel replication, ImageNet-pretrained backbones, and four-view TTA. This notebook uses fixed ImageNet normalization constants; these are not estimated from outer-test data. Training augmentation is applied only to the inner-training images.

Patient-level probability is the **mean** of all image probabilities belonging to the corrected patient ID.

In [4]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def read_grayscale(path):
    ext = Path(path).suffix.lower()

    if ext == ".dcm":
        try:
            import pydicom
        except ImportError as e:
            raise ImportError("DICOM detected. Install pydicom: pip install pydicom") from e
        ds = pydicom.dcmread(path)
        img = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img
        lo, hi = np.percentile(img, [0.5, 99.5])
        if hi <= lo:
            lo, hi = float(img.min()), float(img.max())
        img = np.clip((img - lo) / max(hi - lo, 1e-8), 0, 1)
        return img.astype(np.float32)

    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img.astype(np.float32) / 255.0


def preprocess_image(path, size=IMG_SIZE):
    img = read_grayscale(path)
    mask_source = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    _, mask = cv2.threshold(mask_source, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        if w > 20 and h > 20:
            img = img[y:y+h, x:x+w]

    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    img = np.stack([img, img, img], axis=0).astype(np.float32)
    return img


class MammoDataset(Dataset):
    def __init__(self, dataframe, train=False, size=IMG_SIZE):
        self.df = dataframe.reset_index(drop=True)
        self.train = train
        self.size = size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = preprocess_image(row["path"], self.size)

        if self.train:
            if random.random() < 0.5:
                img = img[:, :, ::-1].copy()
            if random.random() < 0.25:
                k = random.choice([1, 3])
                img = np.rot90(img, k=k, axes=(1, 2)).copy()
            if random.random() < 0.30:
                factor = random.uniform(0.90, 1.10)
                img = np.clip(img * factor, 0, 1)

        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return (
            torch.from_numpy(img.astype(np.float32, copy=False)),
            torch.tensor(float(row["label"]), dtype=torch.float32),
            str(row["patient_id"]),
        )


def build_model(backbone_name, device=DEVICE):
    model = timm.create_model(
        backbone_name,
        pretrained=True,
        num_classes=1,
        in_chans=3,
    )
    return model.to(device)


def aggregate_patient_predictions(image_pred_df, prob_col="prob", agg=PATIENT_AGG):
    if agg not in {"mean", "max"}:
        raise ValueError("PATIENT_AGG must be 'mean' or 'max'.")

    grouped = image_pred_df.groupby("patient_id")
    label_check = grouped["label"].nunique()
    if (label_check > 1).any():
        raise RuntimeError("A patient has inconsistent image labels.")

    return (
        grouped.agg(
            y=("label", "max"),
            p=(prob_col, agg),
            n_images=("path", "size"),
        )
        .reset_index()
    )


@torch.no_grad()
def predict_images(model, dataframe, tta=False, seed=0):
    model.eval()
    ds = MammoDataset(dataframe, train=False)
    loader = make_loader(ds, BATCH_SIZE, shuffle=False, seed=seed)

    tta_transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),       # horizontal
        lambda x: torch.flip(x, dims=[2]),       # vertical
        lambda x: torch.rot90(x, k=2, dims=[2, 3]),
    ] if tta else [lambda x: x]

    probs, labels, patients = [], [], []

    for imgs, y, pids in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        p = torch.zeros(imgs.size(0), device=DEVICE)

        for tfm in tta_transforms:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                logits = model(tfm(imgs)).squeeze(1)
            p += torch.sigmoid(logits)

        p /= len(tta_transforms)
        probs.extend(p.float().cpu().numpy().tolist())
        labels.extend(y.numpy().astype(int).tolist())
        patients.extend(list(pids))

    out = dataframe[["path", "patient_id", "label"]].reset_index(drop=True).copy()
    out["prob"] = np.asarray(probs, dtype=float)

    if not np.array_equal(out["label"].astype(int).values, np.asarray(labels).astype(int)):
        raise RuntimeError("Prediction/label order mismatch.")
    if not np.array_equal(out["patient_id"].astype(str).values, np.asarray(patients).astype(str)):
        raise RuntimeError("Prediction/patient order mismatch.")

    return out

## 4. Patient-grouped 3-fold outer splits, multiple seeds, and balance audit

The split is created at the **patient level**, then mapped back to all images for that patient. The main experiment remains 3-fold. The 5- and 10-fold block below checks only class-balance feasibility; it does not replace the submitted 3-fold experiment.

In [5]:
def patient_table_from_df(dataframe):
    return (
        dataframe.groupby("patient_id")
        .agg(label=("label", "max"), n_images=("path", "size"))
        .reset_index()
    )


def build_outer_folds(dataframe, n_splits, seed):
    pt = patient_table_from_df(dataframe)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_map = {}

    for fold, (_, test_idx) in enumerate(skf.split(pt["patient_id"], pt["label"])):
        for pid in pt.iloc[test_idx]["patient_id"].astype(str):
            fold_map[pid] = fold

    out = dataframe.copy()
    out["fold"] = out["patient_id"].astype(str).map(fold_map)
    if out["fold"].isna().any():
        raise RuntimeError("Some images were not assigned to an outer fold.")
    out["fold"] = out["fold"].astype(int)
    return out


def verify_outer_split(split_df, n_splits):
    rows = []
    for fold in range(n_splits):
        train_ids = set(split_df.loc[split_df["fold"] != fold, "patient_id"].astype(str))
        test_ids = set(split_df.loc[split_df["fold"] == fold, "patient_id"].astype(str))
        overlap = train_ids & test_ids
        if overlap:
            raise RuntimeError(f"Patient leakage in fold {fold}: {list(overlap)[:5]}")
        rows.append({
            "fold": fold,
            "train_patients": len(train_ids),
            "test_patients": len(test_ids),
            "overlap": 0,
        })
    return pd.DataFrame(rows)


def fold_balance_table(split_df, seed):
    p = (
        split_df.groupby(["fold", "patient_id"])
        .agg(label=("label", "max"), n_images=("path", "size"))
        .reset_index()
    )
    out = (
        p.groupby(["fold", "label"])
        .agg(patients=("patient_id", "nunique"), images=("n_images", "sum"))
        .reset_index()
    )
    out["seed"] = seed
    out["class"] = out["label"].map({0: "Not-Cancer", 1: "Cancer"})
    return out


split_frames = {}
balance_frames = []
leakage_frames = []

for seed in SEEDS:
    sdf = build_outer_folds(df, N_FOLDS, seed)
    split_frames[seed] = sdf

    sdf[[
        "path", "filename", "raw_patient_id", "patient_id", "class_folder",
        "label", "side", "view", "fold"
    ]].to_csv(OUTPUT_DIR / f"split_seed_{seed}.csv", index=False)

    bal = fold_balance_table(sdf, seed)
    balance_frames.append(bal)

    leak = verify_outer_split(sdf, N_FOLDS)
    leak["seed"] = seed
    leakage_frames.append(leak)

fold_balance = pd.concat(balance_frames, ignore_index=True)
leakage_audit = pd.concat(leakage_frames, ignore_index=True)
fold_balance.to_csv(OUTPUT_DIR / "fold_class_balance.csv", index=False)
leakage_audit.to_csv(OUTPUT_DIR / "leakage_audit.csv", index=False)

display(fold_balance)
display(leakage_audit)


# Reviewer-requested justification support: balance-only sensitivity for k=3/5/10.
def split_balance_sensitivity(dataframe, k_values=(3, 5, 10), seeds=SEEDS):
    pt = patient_table_from_df(dataframe)
    rows = []
    class_counts = pt["label"].value_counts().to_dict()

    for k in k_values:
        feasible = min(class_counts.values()) >= k
        if not feasible:
            rows.append({"k": k, "seed": np.nan, "feasible": False})
            continue

        for seed in seeds:
            skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
            fractions, cancer_counts, total_counts = [], [], []
            for _, idx in skf.split(pt["patient_id"], pt["label"]):
                fold_pt = pt.iloc[idx]
                fractions.append(float(fold_pt["label"].mean()))
                cancer_counts.append(int((fold_pt["label"] == 1).sum()))
                total_counts.append(len(fold_pt))

            rows.append({
                "k": k,
                "seed": seed,
                "feasible": True,
                "min_patients_per_fold": min(total_counts),
                "max_patients_per_fold": max(total_counts),
                "min_cancer_patients": min(cancer_counts),
                "max_cancer_patients": max(cancer_counts),
                "mean_cancer_fraction": float(np.mean(fractions)),
                "sd_cancer_fraction": float(np.std(fractions, ddof=1)),
            })

    return pd.DataFrame(rows)


k_balance = split_balance_sensitivity(df)
k_balance.to_csv(OUTPUT_DIR / "kfold_balance_sensitivity.csv", index=False)
display(k_balance)

,fold,label,patients,images,seed,class
0,0,0,2030,2030,42,Not-Cancer
1,0,1,1198,1198,42,Cancer
2,1,0,2029,2029,42,Not-Cancer
3,1,1,1199,1199,42,Cancer
4,2,0,2029,2029,42,Not-Cancer
5,2,1,1199,1199,42,Cancer
6,0,0,2030,2030,123,Not-Cancer
7,0,1,1198,1198,123,Cancer
8,1,0,2029,2029,123,Not-Cancer
9,1,1,1199,1199,123,Cancer


,fold,train_patients,test_patients,overlap,seed
0,0,6456,3228,0,42
1,1,6456,3228,0,42
2,2,6456,3228,0,42
3,0,6456,3228,0,123
4,1,6456,3228,0,123
5,2,6456,3228,0,123
6,0,6456,3228,0,2026
7,1,6456,3228,0,2026
8,2,6456,3228,0,2026


,k,seed,feasible,min_patients_per_fold,max_patients_per_fold,min_cancer_patients,max_cancer_patients,mean_cancer_fraction,sd_cancer_fraction
0,3,42,True,3228,3228,1198,1199,0.371334,0.000179
1,3,123,True,3228,3228,1198,1199,0.371334,0.000179
2,3,2026,True,3228,3228,1198,1199,0.371334,0.000179
3,5,42,True,1936,1937,719,720,0.371334,0.000225
4,5,123,True,1936,1937,719,720,0.371334,0.000225
5,5,2026,True,1936,1937,719,720,0.371334,0.000225
6,10,42,True,968,969,359,360,0.371334,0.000428
7,10,123,True,968,969,359,360,0.371334,0.000428
8,10,2026,True,968,969,359,360,0.371334,0.000428


## 5. Inner patient split and metric helpers

Every outer training fold is split again at the patient level. This inner validation set is used for early stopping, best-single selection, optional exploratory ensemble weighting, Platt calibration, and Youden thresholds. The outer test fold remains untouched until final prediction.

In [6]:
def make_inner_split(outer_train_df, seed, val_fraction=INNER_VAL_FRACTION):
    pt = patient_table_from_df(outer_train_df)
    train_ids, val_ids = train_test_split(
        pt["patient_id"].astype(str).values,
        test_size=val_fraction,
        random_state=seed,
        stratify=pt["label"].values,
    )
    train_ids, val_ids = set(train_ids), set(val_ids)

    if train_ids & val_ids:
        raise RuntimeError("Inner patient leakage detected.")

    tr = outer_train_df[outer_train_df["patient_id"].astype(str).isin(train_ids)].reset_index(drop=True)
    va = outer_train_df[outer_train_df["patient_id"].astype(str).isin(val_ids)].reset_index(drop=True)

    if tr["label"].nunique() < 2 or va["label"].nunique() < 2:
        raise RuntimeError("Inner split lost a class. Adjust INNER_VAL_FRACTION or seed.")

    return tr, va


def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan


def safe_ap(y, p):
    return float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan


def threshold_metrics_from_pred(y, pred):
    y = np.asarray(y).astype(int)
    pred = np.asarray(pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    def div(a, b):
        return float(a / b) if b else np.nan

    return {
        "accuracy": div(tp + tn, tp + tn + fp + fn),
        "sensitivity": div(tp, tp + fn),
        "specificity": div(tn, tn + fp),
        "ppv": div(tp, tp + fp),
        "npv": div(tn, tn + fn),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y)

    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (p >= lo) & (p <= hi) if i == n_bins - 1 else (p >= lo) & (p < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(float(y[mask].mean()) - float(p[mask].mean()))
    return float(ece)


def youden_threshold(y, p):
    fpr, tpr, thresholds = roc_curve(y, p)
    valid = np.isfinite(thresholds)
    fpr, tpr, thresholds = fpr[valid], tpr[valid], thresholds[valid]
    j = tpr - fpr
    best = np.max(j)
    candidates = np.flatnonzero(np.isclose(j, best))
    idx = candidates[np.argmin(np.abs(thresholds[candidates] - 0.5))]
    return float(thresholds[idx]), float(best)


class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(solver="lbfgs", max_iter=1000)

    @staticmethod
    def _logit_feature(p):
        p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p)).reshape(-1, 1)

    def fit(self, p, y):
        self.model.fit(self._logit_feature(p), np.asarray(y).astype(int))
        return self

    def predict(self, p):
        return self.model.predict_proba(self._logit_feature(p))[:, 1]


def align_patient_tables(pred_by_backbone):
    merged = None
    for bb, img_pred in pred_by_backbone.items():
        pat = aggregate_patient_predictions(img_pred)
        pat = pat.rename(columns={"p": f"p_{bb}", "n_images": f"n_images_{bb}"})
        keep = ["patient_id", "y", f"p_{bb}", f"n_images_{bb}"]
        if merged is None:
            merged = pat[keep].copy()
        else:
            merged = merged.merge(
                pat[keep], on=["patient_id", "y"], how="inner", validate="one_to_one"
            )
    return merged.sort_values("patient_id").reset_index(drop=True)


def choose_best_backbone(inner_pat):
    rows = []
    for bb in BACKBONES:
        rows.append({
            "backbone": bb,
            "roc_auc": safe_auc(inner_pat["y"], inner_pat[f"p_{bb}"]),
            "pr_auc": safe_ap(inner_pat["y"], inner_pat[f"p_{bb}"]),
        })
    table = pd.DataFrame(rows).sort_values(["roc_auc", "pr_auc"], ascending=False)
    return str(table.iloc[0]["backbone"]), table


def tune_exploratory_weight(inner_pat):
    b1, b2 = BACKBONES
    rows = []
    for w in WEIGHT_GRID:
        p = w * inner_pat[f"p_{b1}"].values + (1 - w) * inner_pat[f"p_{b2}"].values
        rows.append({
            "w_backbone1": float(w),
            "roc_auc": safe_auc(inner_pat["y"], p),
            "pr_auc": safe_ap(inner_pat["y"], p),
        })
    table = pd.DataFrame(rows).sort_values(["pr_auc", "roc_auc"], ascending=False)
    return table.iloc[0].to_dict(), table

## 6. Paper-aligned training function

Checkpoint selection uses **inner-validation patient ROC-AUC without TTA**, matching the submitted paper's description that per-fold validation AUCs used for model selection were reported without TTA. Final inner-validation and outer-test predictions are then generated with the same four-view TTA for every backbone and ensemble comparison.

In [7]:
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    seen = 0

    for step, (imgs, labels, _) in enumerate(loader, start=1):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(loader))
        if should_step:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        batch_n = len(labels)
        running_loss += float(loss.item()) * GRAD_ACCUM_STEPS * batch_n
        seen += batch_n

    return running_loss / max(seen, 1)


def train_one_backbone_fold(backbone_name, inner_train_df, inner_val_df, outer_test_df, seed, fold):
    run_seed = seed * 100 + fold * 10 + BACKBONES.index(backbone_name)
    seed_everything(run_seed)

    train_ds = MammoDataset(inner_train_df, train=True)
    train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, seed=run_seed)

    model = build_model(backbone_name)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BACKBONE_LR[backbone_name],
        weight_decay=1e-5,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    max_epochs = BACKBONE_MAX_EPOCHS[backbone_name]
    patience = BACKBONE_PATIENCE[backbone_name]
    ckpt = CHECKPOINT_DIR / f"{backbone_name}_seed{seed}_fold{fold}.pt"

    best_auc = -np.inf
    best_ap = np.nan
    best_epoch = -1
    stale = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, scaler)

        # No-TTA validation for checkpoint selection, as in the submitted paper.
        val_img = predict_images(model, inner_val_df, tta=False, seed=run_seed + epoch)
        val_pat = aggregate_patient_predictions(val_img)
        val_auc = safe_auc(val_pat["y"], val_pat["p"])
        val_ap = safe_ap(val_pat["y"], val_pat["p"])

        history.append({
            "seed": seed,
            "fold": fold,
            "backbone": backbone_name,
            "epoch": epoch,
            "train_loss": loss,
            "inner_val_auc_no_tta": val_auc,
            "inner_val_pr_auc_no_tta": val_ap,
        })

        improved = (best_epoch < 0) or (val_auc > best_auc + EARLY_STOP_MIN_DELTA)
        print(
            f"seed={seed} fold={fold} {backbone_name} "
            f"epoch={epoch:02d}/{max_epochs} loss={loss:.4f} "
            f"innerAUC(noTTA)={val_auc:.4f}" + (" *best*" if improved else "")
        )

        if improved:
            best_auc = float(val_auc)
            best_ap = float(val_ap)
            best_epoch = int(epoch)
            torch.save(model.state_dict(), ckpt)
            stale = 0
        else:
            stale += 1

        if stale >= patience:
            print(f"Early stop: patience={patience}")
            break

    if not ckpt.exists():
        raise RuntimeError(f"Checkpoint was not created: {ckpt}")

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    # Identical TTA for inner comparison/tuning and untouched outer evaluation.
    inner_tta = predict_images(model, inner_val_df, tta=True, seed=run_seed + 9000)
    outer_tta = predict_images(model, outer_test_df, tta=True, seed=run_seed + 10000)

    summary = {
        "seed": seed,
        "fold": fold,
        "backbone": backbone_name,
        "lr": BACKBONE_LR[backbone_name],
        "max_epochs": max_epochs,
        "patience": patience,
        "best_epoch": best_epoch,
        "best_inner_auc_no_tta": best_auc,
        "best_inner_pr_auc_no_tta": best_ap,
        "checkpoint": str(ckpt),
    }

    del model, train_loader, train_ds
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return inner_tta, outer_tta, summary, pd.DataFrame(history)

## 7. Main experiment: 3 folds × 3 seeds × 2 backbones

The **submitted primary ensemble** is `equal_avg`: 50/50 average of the two patient-level TTA probabilities. The validation-weighted ensemble is saved only as an exploratory reviewer comparison and does not replace the submitted method.

In [ ]:
all_outer_rows = []
all_tuning_rows = []
all_training_summaries = []
all_histories = []

for seed in SEEDS:
    print("\n" + "=" * 100)
    print("SEED", seed)
    print("=" * 100)
    split_df = split_frames[seed]

    for fold in range(N_FOLDS):
        print("\n" + "-" * 100)
        print(f"Seed {seed} | Outer fold {fold}")
        print("-" * 100)

        outer_train = split_df[split_df["fold"] != fold].reset_index(drop=True)
        outer_test = split_df[split_df["fold"] == fold].reset_index(drop=True)

        inner_seed = seed + 1000 * (fold + 1)
        inner_train, inner_val = make_inner_split(outer_train, inner_seed)

        p_train = set(inner_train["patient_id"].astype(str))
        p_val = set(inner_val["patient_id"].astype(str))
        p_test = set(outer_test["patient_id"].astype(str))
        if (p_train & p_val) or (p_train & p_test) or (p_val & p_test):
            raise RuntimeError("Inner/outer patient leakage detected.")

        print(
            "Patients -> inner train:", len(p_train),
            "inner val:", len(p_val),
            "outer test:", len(p_test),
        )

        inner_preds = {}
        outer_preds = {}

        for bb in BACKBONES:
            inner_tta, outer_tta, summary, hist = train_one_backbone_fold(
                bb, inner_train, inner_val, outer_test, seed, fold
            )
            inner_preds[bb] = inner_tta
            outer_preds[bb] = outer_tta
            all_training_summaries.append(summary)
            all_histories.append(hist)

        inner_pat = align_patient_tables(inner_preds)
        outer_pat = align_patient_tables(outer_preds)

        b1, b2 = BACKBONES

        # Submitted primary ensemble: unweighted averaging.
        for frame in (inner_pat, outer_pat):
            frame["p_equal_avg_raw"] = 0.5 * frame[f"p_{b1}"] + 0.5 * frame[f"p_{b2}"]

        # Single-best backbone selected only on inner validation.
        best_bb, best_bb_table = choose_best_backbone(inner_pat)
        for frame in (inner_pat, outer_pat):
            frame["p_single_best_raw"] = frame[f"p_{best_bb}"]

        # Exploratory reviewer comparison: validation-tuned weighted average.
        best_weight_info, weight_table = tune_exploratory_weight(inner_pat)
        w = float(best_weight_info["w_backbone1"])
        for frame in (inner_pat, outer_pat):
            frame["p_weighted_exploratory_raw"] = (
                w * frame[f"p_{b1}"] + (1 - w) * frame[f"p_{b2}"]
            )

        # Fixed backbone aliases make downstream comparison explicit.
        for frame in (inner_pat, outer_pat):
            frame["p_effnet_raw"] = frame[f"p_{BACKBONES[0]}"]
            frame["p_convnext_raw"] = frame[f"p_{BACKBONES[1]}"]

        method_cols = {
            "effnet_b4": "p_effnet_raw",
            "convnextv2_tiny": "p_convnext_raw",
            "single_best": "p_single_best_raw",
            "equal_avg_primary": "p_equal_avg_raw",
            "weighted_exploratory": "p_weighted_exploratory_raw",
        }

        # Fit calibration and choose Youden threshold on inner validation only.
        threshold_info = {}
        for method, raw_col in method_cols.items():
            cal = PlattCalibrator().fit(inner_pat[raw_col], inner_pat["y"])
            inner_pat[f"p_{method}_cal"] = cal.predict(inner_pat[raw_col])
            outer_pat[f"p_{method}_cal"] = cal.predict(outer_pat[raw_col])

            thr, j = youden_threshold(inner_pat["y"], inner_pat[raw_col])
            threshold_info[method] = (thr, j)
            outer_pat[f"youden_{method}"] = thr
            outer_pat[f"pred_{method}_default"] = (outer_pat[raw_col] >= 0.5).astype(int)
            outer_pat[f"pred_{method}_youden"] = (outer_pat[raw_col] >= thr).astype(int)

        outer_pat["seed"] = seed
        outer_pat["fold"] = fold
        outer_pat["best_backbone"] = best_bb
        outer_pat["exploratory_weight_effnet"] = w
        outer_pat["exploratory_weight_convnext"] = 1 - w

        all_outer_rows.append(outer_pat)

        tuning_row = {
            "seed": seed,
            "fold": fold,
            "inner_train_patients": len(p_train),
            "inner_val_patients": len(p_val),
            "outer_test_patients": len(p_test),
            "best_backbone": best_bb,
            "exploratory_weight_effnet": w,
            "exploratory_weight_convnext": 1 - w,
        }
        for method, (thr, j) in threshold_info.items():
            tuning_row[f"youden_{method}"] = thr
            tuning_row[f"youden_j_{method}"] = j

        all_tuning_rows.append(tuning_row)
        best_bb_table.to_csv(OUTPUT_DIR / f"best_backbone_seed{seed}_fold{fold}.csv", index=False)
        weight_table.to_csv(OUTPUT_DIR / f"exploratory_weight_search_seed{seed}_fold{fold}.csv", index=False)

outer_patient = pd.concat(all_outer_rows, ignore_index=True)
tuning_table = pd.DataFrame(all_tuning_rows)
training_summary = pd.DataFrame(all_training_summaries)
training_history = pd.concat(all_histories, ignore_index=True)

# Each seed must contain one outer prediction per unique patient.
expected_patients = df["patient_id"].nunique()
for seed in SEEDS:
    s = outer_patient[outer_patient["seed"] == seed]
    if s["patient_id"].duplicated().any():
        raise RuntimeError(f"Duplicate outer patient prediction rows for seed {seed}.")
    if len(s) != expected_patients:
        raise RuntimeError(
            f"Seed {seed}: expected {expected_patients} patient rows, got {len(s)}."
        )

outer_patient.to_csv(OUTPUT_DIR / "outer_patient_predictions_all_seeds.csv", index=False)
tuning_table.to_csv(OUTPUT_DIR / "inner_tuning_choices.csv", index=False)
training_summary.to_csv(OUTPUT_DIR / "training_summary.csv", index=False)
training_history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

display(tuning_table)


SEED 42

----------------------------------------------------------------------------------------------------
Seed 42 | Outer fold 0
----------------------------------------------------------------------------------------------------
Patients -> inner train: 5681 inner val: 775 outer test: 3228


/tmp/ipykernel_595169/344769596.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


seed=42 fold=0 tf_efficientnet_b4 epoch=01/25 loss=0.8938 innerAUC(noTTA)=0.7625 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=02/25 loss=0.5486 innerAUC(noTTA)=0.8120 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=03/25 loss=0.4654 innerAUC(noTTA)=0.8103
seed=42 fold=0 tf_efficientnet_b4 epoch=04/25 loss=0.4001 innerAUC(noTTA)=0.8355 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=05/25 loss=0.3533 innerAUC(noTTA)=0.8512 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=06/25 loss=0.3155 innerAUC(noTTA)=0.8417
seed=42 fold=0 tf_efficientnet_b4 epoch=07/25 loss=0.2864 innerAUC(noTTA)=0.8561 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=08/25 loss=0.2644 innerAUC(noTTA)=0.8719 *best*
seed=42 fold=0 tf_efficientnet_b4 epoch=09/25 loss=0.2370 innerAUC(noTTA)=0.8518
seed=42 fold=0 tf_efficientnet_b4 epoch=10/25 loss=0.2067 innerAUC(noTTA)=0.8537
seed=42 fold=0 tf_efficientnet_b4 epoch=11/25 loss=0.2010 innerAUC(noTTA)=0.8528
seed=42 fold=0 tf_efficientnet_b4 epoch=12/25 loss=0.1850 innerAUC(

/tmp/ipykernel_595169/344769596.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


seed=42 fold=0 convnextv2_tiny epoch=01/17 loss=0.6068 innerAUC(noTTA)=0.7079 *best*
seed=42 fold=0 convnextv2_tiny epoch=02/17 loss=0.5617 innerAUC(noTTA)=0.7514 *best*


## 8. Reviewer metrics and patient-bootstrap 95% confidence intervals

Threshold metrics are calculated from **raw, uncalibrated model probabilities** to stay aligned with the submitted 0.5 threshold. Calibration is reported separately. The Youden threshold for each fold/method comes only from its inner validation patients.

In [ ]:
METHODS = {
    "effnet_b4": "p_effnet_raw",
    "convnextv2_tiny": "p_convnext_raw",
    "single_best": "p_single_best_raw",
    "equal_avg_primary": "p_equal_avg_raw",
    "weighted_exploratory": "p_weighted_exploratory_raw",
}


def bootstrap_ci_metric(rows, metric_fn, n_boot=N_BOOT, seed=BOOTSTRAP_SEED):
    rng = np.random.RandomState(seed)
    n = len(rows)
    point = metric_fn(rows)
    vals = []

    for _ in range(n_boot):
        sample = rows.iloc[rng.randint(0, n, size=n)]
        if sample["y"].nunique() < 2:
            continue
        try:
            v = float(metric_fn(sample))
            if np.isfinite(v):
                vals.append(v)
        except Exception:
            pass

    if not vals:
        return float(point), np.nan, np.nan
    return (
        float(point),
        float(np.percentile(vals, 2.5)),
        float(np.percentile(vals, 97.5)),
    )


seed_metric_rows = []
ci_rows = []

for seed in SEEDS:
    sdf = outer_patient[outer_patient["seed"] == seed].copy()

    for method, raw_col in METHODS.items():
        cal_col = f"p_{method}_cal"
        y = sdf["y"].values
        raw_p = sdf[raw_col].values
        cal_p = sdf[cal_col].values

        default_m = threshold_metrics_from_pred(y, sdf[f"pred_{method}_default"].values)
        youden_m = threshold_metrics_from_pred(y, sdf[f"pred_{method}_youden"].values)

        row = {
            "seed": seed,
            "method": method,
            "roc_auc": safe_auc(y, raw_p),
            "pr_auc": safe_ap(y, raw_p),
            "brier_raw": float(brier_score_loss(y, raw_p)),
            "ece_raw": expected_calibration_error(y, raw_p),
            "brier_platt": float(brier_score_loss(y, cal_p)),
            "ece_platt": expected_calibration_error(y, cal_p),
            **{f"default_{k}": v for k, v in default_m.items()},
            **{f"youden_{k}": v for k, v in youden_m.items()},
        }
        seed_metric_rows.append(row)

        metric_fns = {
            "roc_auc": lambda d, c=raw_col: safe_auc(d["y"], d[c]),
            "pr_auc": lambda d, c=raw_col: safe_ap(d["y"], d[c]),
            "brier_raw": lambda d, c=raw_col: brier_score_loss(d["y"], d[c]),
            "default_accuracy": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_default"])["accuracy"],
            "default_sensitivity": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_default"])["sensitivity"],
            "default_specificity": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_default"])["specificity"],
            "default_ppv": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_default"])["ppv"],
            "default_npv": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_default"])["npv"],
            "youden_accuracy": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_youden"])["accuracy"],
            "youden_sensitivity": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_youden"])["sensitivity"],
            "youden_specificity": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_youden"])["specificity"],
            "youden_ppv": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_youden"])["ppv"],
            "youden_npv": lambda d, m=method: threshold_metrics_from_pred(d["y"], d[f"pred_{m}_youden"])["npv"],
        }

        for metric_name, fn in metric_fns.items():
            point, lo, hi = bootstrap_ci_metric(
                sdf, fn, n_boot=N_BOOT, seed=BOOTSTRAP_SEED + seed
            )
            ci_rows.append({
                "seed": seed,
                "method": method,
                "metric": metric_name,
                "point": point,
                "ci_low": lo,
                "ci_high": hi,
            })

seed_metrics = pd.DataFrame(seed_metric_rows)
bootstrap_cis = pd.DataFrame(ci_rows)
seed_metrics.to_csv(OUTPUT_DIR / "metrics_by_seed.csv", index=False)
bootstrap_cis.to_csv(OUTPUT_DIR / "patient_bootstrap_95ci.csv", index=False)


def mean_ci_across_seeds(values, confidence=0.95):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    mean = float(np.mean(x))
    if len(x) < 2:
        return mean, np.nan, np.nan, np.nan
    sd = float(np.std(x, ddof=1))
    se = sd / math.sqrt(len(x))
    crit = float(student_t.ppf((1 + confidence) / 2, df=len(x) - 1)) if student_t is not None else 1.96
    return mean, sd, mean - crit * se, mean + crit * se


summary_rows = []
summary_metrics = [
    "roc_auc", "pr_auc", "brier_raw", "ece_raw", "brier_platt", "ece_platt",
    "default_accuracy", "default_sensitivity", "default_specificity", "default_ppv", "default_npv",
    "youden_accuracy", "youden_sensitivity", "youden_specificity", "youden_ppv", "youden_npv",
]

for method in METHODS:
    mdf = seed_metrics[seed_metrics["method"] == method]
    for metric in summary_metrics:
        mean, sd, lo, hi = mean_ci_across_seeds(mdf[metric].values)
        summary_rows.append({
            "method": method,
            "metric": metric,
            "mean": mean,
            "sd_across_seeds": sd,
            "seed_mean_ci_low": lo,
            "seed_mean_ci_high": hi,
            "n_seeds": len(mdf),
        })

seed_summary = pd.DataFrame(summary_rows)
seed_summary.to_csv(OUTPUT_DIR / "across_seed_summary.csv", index=False)

# Reviewer-facing compact comparison.
comparison = seed_summary[
    seed_summary["metric"].isin([
        "roc_auc", "pr_auc", "default_accuracy", "default_sensitivity",
        "default_specificity", "default_ppv", "default_npv",
        "brier_raw", "ece_raw"
    ])
].copy()
comparison.to_csv(OUTPUT_DIR / "reviewer_baseline_comparison.csv", index=False)

display(seed_metrics)
display(comparison)

## 9. Sanity check: confusion matrices must now be patient-level

For each seed, the sum of TN+FP+FN+TP for the primary ensemble must equal the number of unique corrected patients—not the 9,684 image count.

In [ ]:
patient_level_sanity = []
for seed in SEEDS:
    sdf = outer_patient[outer_patient["seed"] == seed]
    m_default = threshold_metrics_from_pred(
        sdf["y"], sdf["pred_equal_avg_primary_default"]
    )
    cm_total = m_default["tn"] + m_default["fp"] + m_default["fn"] + m_default["tp"]
    expected = df["patient_id"].nunique()
    if cm_total != expected:
        raise RuntimeError(
            f"Seed {seed}: confusion matrix total {cm_total} != unique patients {expected}."
        )
    patient_level_sanity.append({
        "seed": seed,
        "confusion_matrix_total": cm_total,
        "unique_patients": expected,
        **m_default,
    })

patient_level_sanity = pd.DataFrame(patient_level_sanity)
patient_level_sanity.to_csv(OUTPUT_DIR / "patient_level_confusion_sanity.csv", index=False)
display(patient_level_sanity)

## 10. Calibration analysis

Both native/raw and train-side Platt-calibrated probabilities are evaluated. Calibration does not replace the submitted primary threshold results; it is an additional reviewer-requested analysis.

In [ ]:
calibration_rows = []

for method, raw_col in METHODS.items():
    fig, ax = plt.subplots(figsize=(6, 5))

    for seed in SEEDS:
        sdf = outer_patient[outer_patient["seed"] == seed]
        cal_col = f"p_{method}_cal"

        for version, col in [("raw", raw_col), ("platt", cal_col)]:
            frac_pos, mean_pred = calibration_curve(
                sdf["y"], sdf[col], n_bins=10, strategy="quantile"
            )
            calibration_rows.extend([
                {
                    "method": method,
                    "seed": seed,
                    "version": version,
                    "mean_pred": float(mp),
                    "fraction_positive": float(fp),
                }
                for mp, fp in zip(mean_pred, frac_pos)
            ])

        # Plot Platt-calibrated curve per seed for readability.
        frac_pos, mean_pred = calibration_curve(
            sdf["y"], sdf[cal_col], n_bins=10, strategy="quantile"
        )
        ax.plot(mean_pred, frac_pos, marker="o", label=f"seed {seed}")

    ax.plot([0, 1], [0, 1], linestyle="--", label="ideal")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed cancer fraction")
    ax.set_title(f"Patient-level calibration (Platt): {method}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"calibration_{method}.png", dpi=300, bbox_inches="tight")
    plt.show()

pd.DataFrame(calibration_rows).to_csv(OUTPUT_DIR / "calibration_curve_values.csv", index=False)

## 11. Decision-curve analysis

Decision-curve analysis is shown for the **primary equal-average ensemble**, using both its native probability and the train-side Platt-calibrated probability. This is a decision-analytic summary, not evidence that the model is ready for autonomous clinical use.

In [ ]:
def net_benefit(y, p, threshold):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    pred = p >= threshold
    tp = np.sum(pred & (y == 1))
    fp = np.sum(pred & (y == 0))
    n = len(y)
    return (tp / n) - (fp / n) * (threshold / (1 - threshold))


def net_benefit_all(y, threshold):
    prev = float(np.mean(np.asarray(y).astype(int)))
    return prev - (1 - prev) * (threshold / (1 - threshold))


threshold_grid = np.linspace(0.05, 0.80, 76)
dca_rows = []

for seed in SEEDS:
    sdf = outer_patient[outer_patient["seed"] == seed]
    y = sdf["y"].values

    for version, col in [
        ("raw", "p_equal_avg_raw"),
        ("platt", "p_equal_avg_primary_cal"),
    ]:
        p = sdf[col].values
        for pt in threshold_grid:
            dca_rows.append({
                "seed": seed,
                "version": version,
                "threshold": float(pt),
                "net_benefit": float(net_benefit(y, p, pt)),
                "treat_all": float(net_benefit_all(y, pt)),
                "treat_none": 0.0,
            })

dca = pd.DataFrame(dca_rows)
dca.to_csv(OUTPUT_DIR / "decision_curve_values.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
mean_dca = dca.groupby(["version", "threshold"])["net_benefit"].mean().reset_index()
for version in ["raw", "platt"]:
    m = mean_dca[mean_dca["version"] == version]
    ax.plot(m["threshold"], m["net_benefit"], label=f"primary ensemble ({version})")

ref = dca.groupby("threshold")[["treat_all", "treat_none"]].mean().reset_index()
ax.plot(ref["threshold"], ref["treat_all"], linestyle="--", label="treat all")
ax.plot(ref["threshold"], ref["treat_none"], linestyle=":", label="treat none")
ax.set_xlabel("Threshold probability")
ax.set_ylabel("Net benefit")
ax.set_title("Decision-curve analysis — primary patient-level ensemble")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "decision_curve_primary.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. False-negative malignant-case analysis

This analysis uses the **submitted primary equal-average ensemble**. A malignant patient is counted as a false negative at 0.5 when the raw patient-level ensemble score is below 0.5. The Youden decision uses the fold-specific threshold selected from inner validation only.

In [ ]:
malignant = outer_patient[outer_patient["y"] == 1].copy()
malignant["fn_default"] = (malignant["pred_equal_avg_primary_default"] == 0).astype(int)
malignant["fn_youden"] = (malignant["pred_equal_avg_primary_youden"] == 0).astype(int)

fn_seed_records = malignant[(malignant["fn_default"] == 1) | (malignant["fn_youden"] == 1)].copy()
fn_seed_records.to_csv(OUTPUT_DIR / "false_negative_seed_records.csv", index=False)

fn_patient = (
    malignant.groupby("patient_id")
    .agg(
        mean_probability=("p_equal_avg_raw", "mean"),
        sd_probability=("p_equal_avg_raw", "std"),
        fn_rate_default=("fn_default", "mean"),
        fn_rate_youden=("fn_youden", "mean"),
        evaluated_seeds=("seed", "nunique"),
    )
    .reset_index()
)

patient_meta = (
    df.groupby("patient_id")
    .agg(
        n_images=("path", "size"),
        sides=("side", lambda x: "|".join(sorted(set(x)))),
        views=("view", lambda x: "|".join(sorted(set(x)))),
        class_folder=("class_folder", "first"),
    )
    .reset_index()
)

fn_patient = fn_patient.merge(patient_meta, on="patient_id", how="left", validate="one_to_one")
fn_patient["consensus_fn_default"] = (fn_patient["fn_rate_default"] >= 0.5).astype(int)
fn_patient["consensus_fn_youden"] = (fn_patient["fn_rate_youden"] >= 0.5).astype(int)
fn_patient = fn_patient.sort_values(
    ["fn_rate_default", "fn_rate_youden", "mean_probability"],
    ascending=[False, False, True],
)
fn_patient.to_csv(OUTPUT_DIR / "false_negative_patient_analysis.csv", index=False)
display(fn_patient.head(30))


def show_false_negative_gallery(table, source_df, n_patients=8):
    selected = table[
        (table["consensus_fn_default"] == 1) | (table["consensus_fn_youden"] == 1)
    ].head(n_patients)

    if selected.empty:
        print("No consensus false-negative patients found.")
        return

    fig, axes = plt.subplots(len(selected), 1, figsize=(7, 3.1 * len(selected)), squeeze=False)
    for ax, (_, row) in zip(axes[:, 0], selected.iterrows()):
        pid = row["patient_id"]
        image_row = source_df[source_df["patient_id"] == pid].iloc[0]
        img = read_grayscale(image_row["path"])
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        ax.set_title(
            f"{pid} | mean p={row['mean_probability']:.3f} | "
            f"FN@0.5={row['fn_rate_default']:.2f} | FN@Youden={row['fn_rate_youden']:.2f}"
        )

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "false_negative_gallery.png", dpi=300, bbox_inches="tight")
    plt.show()

show_false_negative_gallery(fn_patient, df, n_patients=8)

## 13. Grad-CAM reporting rule

Keep the submitted Grad-CAM section **qualitative** unless a separate radiologist-ROI quantitative experiment is added. It is valid to show representative true-positive, true-negative, false-positive, and false-negative examples, but do not claim localization accuracy or clinical correctness from Grad-CAM alone.

## 14. Optional external validation on CBIS-DDSM or INbreast

Reviewer 2 requested external validation. Set `ENABLE_EXTERNAL_VALIDATION=True` only after preparing `EXTERNAL_CSV` with columns:

- `path` — local PNG/JPG/TIFF/DICOM path;
- `patient_id` — external patient grouping key;
- `label` — `1` malignant/cancer, `0` not-cancer.

The external labels are never used to train or tune Mini-DDSM models. The code below creates a Mini-DDSM development/train and calibration split, trains the two paper backbones, keeps the **equal-average ensemble** as primary, selects the Youden threshold on Mini-DDSM calibration patients, then evaluates the external cohort once.

In [ ]:
def load_external_manifest(csv_path):
    ext = pd.read_csv(csv_path)
    required = {"path", "patient_id", "label"}
    missing = required - set(ext.columns)
    if missing:
        raise ValueError(f"External manifest missing columns: {sorted(missing)}")

    ext = ext.copy()
    ext["path"] = ext["path"].astype(str)
    ext["patient_id"] = ext["patient_id"].astype(str)
    ext["label"] = ext["label"].astype(int)

    if not set(ext["label"].unique()).issubset({0, 1}):
        raise ValueError("External labels must be binary 0/1.")
    if ext.groupby("patient_id")["label"].nunique().max() > 1:
        raise ValueError("An external patient maps to more than one binary label.")
    missing_files = [p for p in ext["path"] if not Path(p).exists()]
    if missing_files:
        raise FileNotFoundError(f"External files missing, first example: {missing_files[0]}")
    return ext


def run_external_validation(mini_df, external_df, seed=42):
    mini_pt = patient_table_from_df(mini_df)
    dev_ids, cal_ids = train_test_split(
        mini_pt["patient_id"].astype(str).values,
        test_size=INNER_VAL_FRACTION,
        random_state=seed,
        stratify=mini_pt["label"].values,
    )
    dev_ids, cal_ids = set(dev_ids), set(cal_ids)
    mini_dev = mini_df[mini_df["patient_id"].astype(str).isin(dev_ids)].reset_index(drop=True)
    mini_cal = mini_df[mini_df["patient_id"].astype(str).isin(cal_ids)].reset_index(drop=True)

    cal_preds, ext_preds, summaries = {}, {}, []
    for bb in BACKBONES:
        cal_tta, ext_tta, summary, _ = train_one_backbone_fold(
            bb, mini_dev, mini_cal, external_df, seed=seed, fold=99
        )
        cal_preds[bb] = cal_tta
        ext_preds[bb] = ext_tta
        summaries.append(summary)

    cal_pat = align_patient_tables(cal_preds)
    ext_pat = align_patient_tables(ext_preds)
    b1, b2 = BACKBONES

    cal_pat["p_equal_avg_raw"] = 0.5 * cal_pat[f"p_{b1}"] + 0.5 * cal_pat[f"p_{b2}"]
    ext_pat["p_equal_avg_raw"] = 0.5 * ext_pat[f"p_{b1}"] + 0.5 * ext_pat[f"p_{b2}"]

    thr, j = youden_threshold(cal_pat["y"], cal_pat["p_equal_avg_raw"])
    calibrator = PlattCalibrator().fit(cal_pat["p_equal_avg_raw"], cal_pat["y"])
    ext_pat["p_equal_avg_primary_cal"] = calibrator.predict(ext_pat["p_equal_avg_raw"])
    ext_pat["pred_default"] = (ext_pat["p_equal_avg_raw"] >= 0.5).astype(int)
    ext_pat["pred_youden"] = (ext_pat["p_equal_avg_raw"] >= thr).astype(int)

    y = ext_pat["y"].values
    raw_p = ext_pat["p_equal_avg_raw"].values
    cal_p = ext_pat["p_equal_avg_primary_cal"].values

    result = {
        "external_dataset": EXTERNAL_NAME,
        "seed": seed,
        "roc_auc": safe_auc(y, raw_p),
        "pr_auc": safe_ap(y, raw_p),
        "brier_raw": float(brier_score_loss(y, raw_p)),
        "ece_raw": expected_calibration_error(y, raw_p),
        "brier_platt": float(brier_score_loss(y, cal_p)),
        "ece_platt": expected_calibration_error(y, cal_p),
        "mini_calibration_youden_threshold": thr,
        "mini_calibration_youden_j": j,
        **{f"default_{k}": v for k, v in threshold_metrics_from_pred(y, ext_pat["pred_default"]).items()},
        **{f"youden_{k}": v for k, v in threshold_metrics_from_pred(y, ext_pat["pred_youden"]).items()},
    }

    return ext_pat, pd.DataFrame([result]), pd.DataFrame(summaries)


if ENABLE_EXTERNAL_VALIDATION:
    external_df = load_external_manifest(EXTERNAL_CSV)
    external_pat, external_metrics, external_training = run_external_validation(
        df, external_df, seed=SEEDS[0]
    )
    external_pat.to_csv(OUTPUT_DIR / f"{EXTERNAL_NAME}_patient_predictions.csv", index=False)
    external_metrics.to_csv(OUTPUT_DIR / f"{EXTERNAL_NAME}_metrics.csv", index=False)
    external_training.to_csv(OUTPUT_DIR / f"{EXTERNAL_NAME}_training_summary.csv", index=False)
    display(external_metrics)
else:
    print("External validation disabled. Add a manifest and set ENABLE_EXTERNAL_VALIDATION=True when ready.")

## 15. Reproducibility export and camera-ready checklist

The outputs generated by this notebook are intended to support the reviewer response and camera-ready rewrite. Do not copy the old submitted image-level-looking confusion-matrix counts into the revised patient-level results. Use the newly generated patient-level outputs only.

In [ ]:
pseudocode = [
    "PATIENT-ID NORMALIZATION / COLLISION CORRECTION",
    "1. Parse the raw patient identifier from each Mini-DDSM filename.",
    "2. Preserve raw_patient_id for audit.",
    '3. Construct corrected patient_id = class_folder + "::" + raw_patient_id.',
    "4. Export raw IDs appearing in more than one class namespace.",
    "5. Keep every image of a corrected patient together.",
    "",
    "OUTER EVALUATION",
    "1. Collapse to one row per corrected patient.",
    "2. Stratify patients into 3 outer folds.",
    "3. Repeat the 3-fold experiment for multiple random seeds.",
    "4. Export exact split files.",
    "",
    "INNER TRAINING / MODEL SELECTION",
    "1. Split only outer-training patients into inner-train and inner-validation.",
    "2. Train each backbone on inner-train only.",
    "3. Select checkpoints using inner-validation patient AUC without TTA.",
    "4. Generate identical four-view TTA predictions for all final comparisons.",
    "5. Primary ensemble = unweighted 50/50 patient-level probability average.",
    "6. Select Youden threshold and fit calibration on inner-validation only.",
    "7. Evaluate outer-test patients once.",
]

(OUTPUT_DIR / "reproducibility_pseudocode.txt").write_text("\n".join(pseudocode) + "\n")

run_config = {
    "DATA_ROOT": str(DATA_ROOT),
    "IMG_SIZE": IMG_SIZE,
    "N_FOLDS": N_FOLDS,
    "SEEDS": SEEDS,
    "INNER_VAL_FRACTION": INNER_VAL_FRACTION,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "NUM_WORKERS": NUM_WORKERS,
    "BACKBONES": BACKBONES,
    "BACKBONE_LR": BACKBONE_LR,
    "BACKBONE_MAX_EPOCHS": BACKBONE_MAX_EPOCHS,
    "BACKBONE_PATIENCE": BACKBONE_PATIENCE,
    "PATIENT_AGG": PATIENT_AGG,
    "N_BOOT": N_BOOT,
    "PRIMARY_ENSEMBLE": "unweighted 50/50 probability average",
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2))

checklist = pd.DataFrame([
    ["Patient-ID collision documented", True, "raw_patient_id_collisions.csv"],
    ["Corrected manifest exported", True, "corrected_patient_manifest.csv"],
    ["Exact split files exported", True, "split_seed_<seed>.csv"],
    ["3-fold repeated across seeds", True, str(SEEDS)],
    ["Patient/image fold balance", True, "fold_class_balance.csv"],
    ["Leakage verification", True, "leakage_audit.csv"],
    ["Primary equal-average ensemble retained", True, "equal_avg_primary"],
    ["Single-backbone identical-TTA comparison", True, "reviewer_baseline_comparison.csv"],
    ["Sensitivity/specificity/PPV/NPV + CIs", True, "patient_bootstrap_95ci.csv"],
    ["Default + train-side Youden thresholds", True, "inner_tuning_choices.csv"],
    ["Calibration", True, "calibration_*.png / metrics_by_seed.csv"],
    ["Decision curve", True, "decision_curve_primary.png"],
    ["False-negative malignant analysis", True, "false_negative_patient_analysis.csv"],
    ["External validation", ENABLE_EXTERNAL_VALIDATION, EXTERNAL_NAME],
], columns=["item", "implemented", "output"])
checklist.to_csv(OUTPUT_DIR / "reviewer_checklist.csv", index=False)
display(checklist)

print("\nAll generated outputs are under:")
print(OUTPUT_DIR.resolve())